In [31]:
import pandas as pd
import numpy as np

data = pd.read_csv('/kaggle/input/mnist-in-csv/mnist_train.csv')
# data_test = pd.read_csv('/kaggle/input/mnist-in-csv/mnist_test.csv')
data


,label,1x1,1x2,1x3,1x4,1x5,1x6,1x7,1x8,1x9,...,28x19,28x20,28x21,28x22,28x23,28x24,28x25,28x26,28x27,28x28
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59996,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59997,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59998,6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [32]:
data = np.array(data)
m, n = data.shape

data_train = data.T
Y = data_train[0]
X = data_train[1:n]
#for avoiding overflow
X = X / 255  
Y

array([5, 0, 4, ..., 5, 6, 8])

In [33]:
def params():
    W1 = np.random.rand(10, 784) - 0.5
    b1 = np.random.rand(10, 1) - 0.5
    W2 = np.random.rand(10, 10) - 0.5
    b2 = np.random.rand(10, 1) - 0.5
    return W1, b1, W2, b2

def ReLU(Z):
    return np.maximum(Z, 0)

def softmax(Z):
    A = np.exp(Z) / sum(np.exp(Z))
    return A

def forward_prop(W1, b1, W2, b2, X):
    Z1 = W1.dot(X) + b1
    A1 = ReLU(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

def ReLU_deriv(Z):
    return Z > 0

def expected(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    one_hot_Y = one_hot_Y.T
    return one_hot_Y

def backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y):
    Y = expected(Y)
    dZ2 = A2 - Y
    dW2 = 1 / m * dZ2.dot(A1.T)
    db2 = 1 / m * np.sum(dZ2)
    dZ1 = W2.T.dot(dZ2) * ReLU_deriv(Z1)
    dW1 = 1 / m * dZ1.dot(X.T)
    db1 = 1 / m * np.sum(dZ1)
    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1    
    W2 = W2 - alpha * dW2  
    b2 = b2 - alpha * db2    
    return W1, b1, W2, b2

In [34]:
def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    print(predictions, Y)
    return np.sum(predictions == Y) / Y.size

def gradient_descent(X, Y, alpha, iterations):
    W1, b1, W2, b2 = params()
    for i in range(iterations):
        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X)
        dW1, db1, dW2, db2 = backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)
        if i % 10 == 0:
            print("Iteration: ", i)
            predictions = get_predictions(A2)
            print(get_accuracy(predictions, Y))
    return W1, b1, W2, b2

In [38]:
W1, b1, W2, b2 = gradient_descent(X_train, Y_train, 0.1, 1000)

Iteration:  0
[8 2 4 ... 8 7 4] [5 0 4 ... 5 6 8]
0.14023333333333332
Iteration:  10
[0 0 4 ... 8 0 4] [5 0 4 ... 5 6 8]
0.23913333333333334
Iteration:  20
[0 0 4 ... 8 0 4] [5 0 4 ... 5 6 8]
0.30205
Iteration:  30
[6 0 4 ... 8 0 4] [5 0 4 ... 5 6 8]
0.34895
Iteration:  40
[2 0 4 ... 8 7 4] [5 0 4 ... 5 6 8]
0.40418333333333334
Iteration:  50
[2 0 4 ... 8 7 4] [5 0 4 ... 5 6 8]
0.4578
Iteration:  60
[2 0 4 ... 5 7 4] [5 0 4 ... 5 6 8]
0.5048833333333334
Iteration:  70
[2 0 4 ... 5 0 4] [5 0 4 ... 5 6 8]
0.5486833333333333
Iteration:  80
[2 0 4 ... 5 0 7] [5 0 4 ... 5 6 8]
0.58835
Iteration:  90
[2 0 4 ... 5 0 7] [5 0 4 ... 5 6 8]
0.6216166666666667
Iteration:  100
[2 0 4 ... 5 0 7] [5 0 4 ... 5 6 8]
0.6496666666666666
Iteration:  110
[3 0 4 ... 5 0 7] [5 0 4 ... 5 6 8]
0.6757166666666666
Iteration:  120
[3 0 4 ... 5 0 7] [5 0 4 ... 5 6 8]
0.6951666666666667
Iteration:  130
[3 0 4 ... 5 0 7] [5 0 4 ... 5 6 8]
0.71265
Iteration:  140
[3 0 4 ... 5 6 7] [5 0 4 ... 5 6 8]
0.7272833333333333

In [44]:
testing_data = pd.read_csv('/kaggle/input/mnist-in-csv/mnist_test.csv')
data_test = np.array(testing_data.T)
Y = data_test[0]
data_test.shape
X = data_test[1:785]
#sam as input varaible during training
X = X / 255 


In [42]:
def predictions(X, W1, b1, W2, b2, Y):
    _, _, _, A2 = forward_prop(W1, b1, W2, b2, X)
    predictions = get_predictions(A2)
    print(get_accuracy(predictions, Y))

In [43]:
predictions(X, W1, b1, W2, b2, Y)

[7 2 1 ... 4 5 6] [7 2 1 ... 4 5 6]
0.8866
